In [ ]:
import pandas as pd
import numpy as np

# 🔹 Load Excel file
file_path = "PLAN_ACTUAL.xlsx"   # change to your file name
df = pd.read_excel(file_path)

# 🔹 Separate material column
materials = df['Material']

# 🔹 Take only demand columns (all except Material)
demand_data = df.drop(columns=['Material'])

# 🔹 Calculate statistics row-wise
mean_demand = demand_data.mean(axis=1)
std_dev = demand_data.std(axis=1, ddof=1)
count = demand_data.count(axis=1)

# 🔹 Standard Error
std_error = std_dev / np.sqrt(count)

# 🔹 95% Confidence Interval
z = 1.96
lower_ci = mean_demand - z * std_error
upper_ci = mean_demand + z * std_error

# 🔹 Create result dataframe
result = pd.DataFrame({
    'Material': materials,
    'Mean_Demand': mean_demand,
    'Std_Dev': std_dev,
    'Lower_95_CI': lower_ci,
    'Upper_95_CI': upper_ci
})

print(result)

# 🔹 Optional: Save to Excel
result.to_excel("Confidence_Interval_Output.xlsx", index=False)


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# 🔹 Load Excel file
file_path = "PLAN_ACTUAL.xlsx"   # change to your file name
df = pd.read_excel(file_path)

# 🔹 Separate material column
materials = df['Material']

# 🔹 Take only demand columns (all except Material)
demand_data = df.drop(columns=['Material'])

# 🔹 Calculate statistics row-wise
mean_demand = demand_data.mean(axis=1)
std_dev = demand_data.std(axis=1, ddof=1)
count = demand_data.count(axis=1)

# 🔹 Standard Error
std_error = std_dev / np.sqrt(count)

# 🔹 95% Confidence Interval using t-score
alpha = 0.05
# t critical value depends on df = count - 1
t_values = stats.t.ppf(1 - alpha/2, df=count - 1)

lower_ci = mean_demand - t_values * std_error
upper_ci = mean_demand + t_values * std_error

# 🔹 Create result dataframe
result = pd.DataFrame({
    'Material': materials,
    'Mean_Demand': mean_demand,
    'Std_Dev': std_dev,
    'Lower_95_CI': lower_ci,
    'Upper_95_CI': upper_ci
})

print(result)

# 🔹 Optional: Save to Excel
result.to_excel("Confidence_Interval_Output.xlsx", index=False)


In [ ]:
import pandas as pd

# =====================================
# LOAD FILES
# =====================================

indent_df = pd.read_excel("indent.xlsx")
actual_df = pd.read_excel("plan_actual.xlsx")

# =====================================
# FIX INDENT (PIVOT STRUCTURE)
# =====================================

indent_df = indent_df.rename(columns={"Row Labels": "Material"})

# Rename "Sum of 1" → "2026-02-01", etc.
new_columns = ["Material"]
for i in range(1, len(indent_df.columns)):
    day = i
    new_columns.append(f"2026-02-{str(day).zfill(2)}")

indent_df.columns = new_columns

# =====================================
# FIX ACTUAL FILE
# =====================================

actual_df.columns = actual_df.columns.str.replace(
    " Total Production Plan", "", regex=False
)

# =====================================
# WIDE → LONG
# =====================================

indent_long = indent_df.melt(
    id_vars=["Material"],
    var_name="Date",
    value_name="Indent_Qty"
)

actual_long = actual_df.melt(
    id_vars=["Material"],
    var_name="Date",
    value_name="Actual_Qty"
)

# Convert to datetime
indent_long["Date"] = pd.to_datetime(indent_long["Date"])
actual_long["Date"] = pd.to_datetime(actual_long["Date"])

# =====================================
# FILTER ACTUAL DATA FOR FEB ONLY
# =====================================

actual_long = actual_long[
    actual_long["Date"] >= "2026-02-01"
]

# =====================================
# MERGE
# =====================================

merged = pd.merge(
    indent_long,
    actual_long,
    on=["Material", "Date"],
    how="inner"
)

# =====================================
# DEVIATION CALCULATION
# =====================================

merged["Deviation"] = merged["Actual_Qty"] - merged["Indent_Qty"]

merged["Deviation_%"] = (
    merged["Deviation"] / merged["Indent_Qty"]
) * 100

# =====================================
# CUMULATIVE FEB DRIFT
# =====================================

merged = merged.sort_values(["Material", "Date"])

merged["Cum_Actual"] = merged.groupby("Material")["Actual_Qty"].cumsum()
merged["Cum_Indent"] = merged.groupby("Material")["Indent_Qty"].cumsum()

merged["Cum_Deviation"] = (
    merged["Cum_Actual"] - merged["Cum_Indent"]
)

# =====================================
# SAVE OUTPUT
# =====================================

merged.to_excel("feb_indent_vs_actual_analysis.xlsx", index=False)

print("February Indent vs Actual comparison completed successfully.")
